# Siniestralidad vial en México: ¿cuándo, dónde y bajo qué condiciones un accidente deja víctimas?

**Proyecto de portafolio · Análisis de datos con Python**
**Fuente:** INEGI — Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS), datos abiertos.

| | |
|---|---|
| **Autor** | *Tu nombre* |
| **Rol** | Analista de datos (proyecto individual: definición del problema, limpieza, análisis, modelado y comunicación) |
| **Herramientas** | Python · pandas · matplotlib · scikit-learn · Jupyter |
| **Periodo analizado** | 2015 al último año publicado |

---
## 1. Problema y objetivo

Cada año se registran en México cientos de miles de accidentes de tránsito en zonas urbanas y suburbanas. La mayoría termina en daños materiales, pero una fracción deja personas heridas o fallecidas. Para quien diseña operativos de tránsito, campañas de prevención o productos de seguros, la pregunta útil no es solo *cuántos* accidentes hay, sino **qué distingue a los que dejan víctimas**.

**Objetivo.** Describir los patrones temporales, geográficos y circunstanciales de los accidentes de tránsito y cuantificar qué factores se asocian con que un accidente deje víctimas (personas heridas o muertas).

**Preguntas guía**
1. ¿Cómo ha evolucionado el número de accidentes y de víctimas desde 2015?
2. ¿En qué días y horas se concentran los accidentes, y en cuáles son más graves?
3. ¿Qué tipos de accidente y qué entidades tienen mayor proporción de accidentes con víctimas?
4. ¿Qué factores (alcohol, cinturón, motocicleta, horario, zona) se asocian con mayor probabilidad de víctimas, controlando por los demás?

## 2. Alcance

| Sí incluye | No incluye (y por qué) |
|---|---|
| Accidentes en zonas urbanas y suburbanas (ATUS), 2015 en adelante | Carreteras federales: las registra la Guardia Nacional, no ATUS |
| Análisis nacional y por entidad federativa | Análisis municipal o georreferenciado: multiplica el trabajo de limpieza |
| Gravedad medida como "accidente con víctimas" | Tasas por habitante o por vehículo: requieren cruzar con población (CONAPO) y parque vehicular |
| Modelo explicativo sencillo (regresión logística) | Modelo predictivo en producción o inferencia causal |

Estas exclusiones mantienen el proyecto terminable en pocos días y quedan anotadas como trabajo futuro en la sección 9.

## 3. Configuración

In [19]:
import io, re, zipfile, unicodedata, urllib.request
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

ANIO_INICIO = 2015                      # alcance temporal del proyecto
DIR_DATOS = Path("datos"); DIR_DATOS.mkdir(exist_ok=True)
DIR_FIG = Path("figuras"); DIR_FIG.mkdir(exist_ok=True)
URL_ATUS = "https://www.inegi.org.mx/contenidos/programas/accidentes/datosabiertos/atus_anual_csv.zip"
ZIP_LOCAL = DIR_DATOS / "atus_anual_csv.zip"

AZUL, ROJO, GRIS = "#2a6f97", "#c1440e", "#8a8f98"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25,
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "left",
})
def guardar(nombre):
    plt.tight_layout(); plt.savefig(DIR_FIG / f"{nombre}.png", bbox_inches="tight"); plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## 4. Obtención de los datos

El INEGI publica ATUS como datos abiertos en un solo ZIP con un CSV por año. El notebook lo descarga una sola vez y lo guarda en `datos/`.
Si la descarga automática falla (red corporativa, cambio de URL), descarga el archivo a mano desde <https://www.inegi.org.mx/programas/accidentes/> → *Datos abiertos* y guárdalo como `datos/atus_anual_csv.zip`.

In [ ]:
if not ZIP_LOCAL.exists():
    print("Descargando ATUS desde INEGI…")
    req = urllib.request.Request(URL_ATUS, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=300) as r, open(ZIP_LOCAL, "wb") as f:
        f.write(r.read())
print(f"Archivo: {ZIP_LOCAL}  ({ZIP_LOCAL.stat().st_size/1e6:.1f} MB)")

def leer_csv(zf, nombre):
    crudo = zf.read(nombre)
    for enc in ("utf-8-sig", "latin-1"):          # la codificación ha cambiado entre ediciones
        try:
            # index_col=False: los CSV 2015–2021 traen una coma final que desalinearía las columnas
            return pd.read_csv(io.BytesIO(crudo), encoding=enc, low_memory=False, index_col=False)
        except UnicodeDecodeError:
            continue

partes = []
with zipfile.ZipFile(ZIP_LOCAL) as zf:
    for nombre in sorted(zf.namelist()):
        m = re.search(r"atus_anual_(\d{4})\.csv$", nombre, flags=re.I)
        if m and int(m.group(1)) >= ANIO_INICIO:   # solo se cargan los años dentro del alcance
            partes.append(leer_csv(zf, nombre))
            print(f"  {nombre}: {len(partes[-1]):,} filas")
assert partes, "No se encontraron archivos atus_anual_AAAA.csv dentro del ZIP: revisa su contenido."
crudo = pd.concat(partes, ignore_index=True)
print(f"\nTotal: {len(crudo):,} filas × {crudo.shape[1]} columnas")
crudo.head()

## 5. Limpieza y preparación

Tres problemas de calidad que hay que resolver antes de analizar:

1. **Nombres de columna distintos según el producto.** Los microdatos (DBF) usan `EDO`, `MPIO`, `HORA`, `EDAD`; los datos abiertos usan `ID_ENTIDAD`, `ID_MUNICIPIO`, `ID_HORA`, `ID_EDAD`. Se unifican.
2. **Códigos vs. etiquetas.** Según la edición, las variables categóricas vienen como texto ("Volcadura") o como clave numérica (5). Se convierten siempre a etiqueta con el catálogo del descriptor de archivos del INEGI.
3. **Valores centinela.** `99` en hora y edad significa "No especificado"; `0` en edad significa "se fugó"; y existen filas de "Certificado cero" (un municipio reporta que *no* tuvo accidentes), que no son accidentes y se eliminan.

In [ ]:
def norm(x):
    # minúsculas sin acentos ni espacios extra, para comparar etiquetas de forma robusta
    x = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", x).strip().lower()

df = crudo.copy()
df.columns = [c.strip().upper() for c in df.columns]
df = df.rename(columns={"EDO": "ID_ENTIDAD", "MPIO": "ID_MUNICIPIO", "HORA": "ID_HORA",
                        "MINUTOS": "ID_MINUTO", "DIA": "ID_DIA", "EDAD": "ID_EDAD"})

CATALOGOS = {
    "DIASEMANA": {0: "Certificado cero", 1: "Lunes", 2: "Martes", 3: "Miércoles", 4: "Jueves",
                  5: "Viernes", 6: "Sábado", 7: "Domingo", 8: "No especificado"},
    "URBANA": {0: "Sin accidente en esta zona", 1: "Accidente en intersección", 2: "Accidente en no intersección"},
    "SUBURBANA": {0: "Sin accidente en esta zona", 1: "Accidente en camino rural",
                  2: "Accidente en carretera estatal", 3: "Accidente en otro camino"},
    "TIPACCID": {0: "Certificado cero", 1: "Colisión con vehículo automotor", 2: "Colisión con peatón (atropellamiento)",
                 3: "Colisión con animal", 4: "Colisión con objeto fijo", 5: "Volcadura", 6: "Caída de pasajero",
                 7: "Salida del camino", 8: "Incendio", 9: "Colisión con ferrocarril",
                 10: "Colisión con motocicleta", 11: "Colisión con ciclista", 12: "Otro"},
    "CAUSAACCI": {1: "Conductor", 2: "Peatón o pasajero", 3: "Falla del vehículo",
                  4: "Mala condición del camino", 5: "Otra"},
    "CAPAROD": {1: "Pavimentada", 2: "No pavimentada"},
    "SEXO": {1: "Se fugó", 2: "Hombre", 3: "Mujer"},
    "ALIENTO": {4: "Sí", 5: "No", 6: "Se ignora"},
    "CINTURON": {7: "Sí", 8: "No", 9: "Se ignora"},
}
def a_etiqueta(s, catalogo):
    num = pd.to_numeric(s, errors="coerce")
    if num.notna().mean() > 0.9:                       # venía como clave numérica
        return num.map(catalogo).fillna("No especificado")
    return s.astype(str).str.strip()                   # ya venía como texto
for col, cat in CATALOGOS.items():
    df[col] = a_etiqueta(df[col], cat)

# --- Fuera los "certificado cero": no son accidentes -----------------------------------------
antes = len(df)
df = df[~df["TIPACCID"].map(norm).str.contains("certificado")].copy()
print(f"Filas de certificado cero eliminadas: {antes - len(df):,}")

# --- Tiempo -----------------------------------------------------------------------------------
MESES = ["enero","febrero","marzo","abril","mayo","junio","julio","agosto","septiembre","octubre","noviembre","diciembre"]
mes = pd.to_numeric(df["MES"], errors="coerce")
if mes.isna().mean() > 0.5:                            # mes como texto
    mes = df["MES"].map(norm).map({m: i + 1 for i, m in enumerate(MESES)})
df["MES"] = mes
df["ANIO"] = pd.to_numeric(df["ANIO"], errors="coerce").astype(int)
df["HORA"] = pd.to_numeric(df["ID_HORA"], errors="coerce").where(lambda h: h.between(0, 23))
ORDEN_DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
df["DIASEMANA"] = df["DIASEMANA"].map(norm).map({norm(d): d for d in ORDEN_DIAS})   # lo demás → NaN
df["DIASEMANA"] = pd.Categorical(df["DIASEMANA"], categories=ORDEN_DIAS, ordered=True)

# --- Entidad ----------------------------------------------------------------------------------
ENTIDADES = ["Aguascalientes","Baja California","Baja California Sur","Campeche","Coahuila","Colima","Chiapas",
    "Chihuahua","Ciudad de México","Durango","Guanajuato","Guerrero","Hidalgo","Jalisco","México","Michoacán",
    "Morelos","Nayarit","Nuevo León","Oaxaca","Puebla","Querétaro","Quintana Roo","San Luis Potosí","Sinaloa",
    "Sonora","Tabasco","Tamaulipas","Tlaxcala","Veracruz","Yucatán","Zacatecas"]
df["ENTIDAD"] = pd.to_numeric(df["ID_ENTIDAD"], errors="coerce").map({i + 1: e for i, e in enumerate(ENTIDADES)})

# --- Víctimas y gravedad ----------------------------------------------------------------------
cols_muertos = [c for c in df.columns if c.endswith("MUERTO")]
cols_heridos = [c for c in df.columns if c.endswith("HERIDO")]
for c in cols_muertos + cols_heridos:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
df["MUERTOS"] = df[cols_muertos].sum(axis=1).astype(int)
df["HERIDOS"] = df[cols_heridos].sum(axis=1).astype(int)
df["GRAVEDAD"] = np.select([df["MUERTOS"] > 0, df["HERIDOS"] > 0], ["Fatal", "No fatal"], "Solo daños")
df["CON_VICTIMAS"] = (df["GRAVEDAD"] != "Solo daños").astype(int)

# --- Variables derivadas ----------------------------------------------------------------------
df["MOTO"] = (pd.to_numeric(df["MOTOCICLET"], errors="coerce").fillna(0) > 0).astype(int)
df["FIN_DE_SEMANA"] = df["DIASEMANA"].isin(["Sábado", "Domingo"]).astype(int)
df["FRANJA"] = pd.cut(df["HORA"], [-1, 5, 11, 17, 23],
                      labels=["Madrugada (0-5)", "Mañana (6-11)", "Tarde (12-17)", "Noche (18-23)"])
df["ZONA"] = np.where(df["URBANA"].map(norm).str.contains("no interseccion"), "Urbana: no intersección",
              np.where(df["URBANA"].map(norm).str.contains("interseccion"), "Urbana: intersección", "Suburbana"))
edad = pd.to_numeric(df["ID_EDAD"], errors="coerce").where(lambda e: e.between(12, 98))
df["GRUPO_EDAD"] = pd.cut(edad, [11, 24, 39, 59, 98], labels=["12-24", "25-39", "40-59", "60+"])
for c in ["ALIENTO", "CINTURON"]:                      # tres valores limpios: Sí / No / Se ignora
    n = df[c].map(norm)
    df[c] = np.where(n.str.startswith("si"), "Sí", np.where(n.str.startswith("no") & ~n.str.contains("especific"), "No", "Se ignora"))

print(f"Base limpia: {len(df):,} accidentes, {df['ANIO'].min()}–{df['ANIO'].max()}")

### Verificaciones de calidad
Antes de sacar conclusiones conviene comprobar que la base se comporta como se espera: cuántos datos faltan en las variables clave y que los totales anuales sean del orden de los que publica el INEGI en sus comunicados (por ejemplo, 340,415 accidentes en 2021).

In [ ]:
calidad = pd.DataFrame({
    "% sin hora": df["HORA"].isna().groupby(df["ANIO"]).mean(),
    "% sin día de la semana": df["DIASEMANA"].isna().groupby(df["ANIO"]).mean(),
    "% sin entidad": df["ENTIDAD"].isna().groupby(df["ANIO"]).mean(),
    "% aliento 'se ignora'": (df["ALIENTO"] == "Se ignora").groupby(df["ANIO"]).mean(),
    "% cinturón 'se ignora'": (df["CINTURON"] == "Se ignora").groupby(df["ANIO"]).mean(),
    "% edad no válida": df["GRUPO_EDAD"].isna().groupby(df["ANIO"]).mean(),
}).mul(100).round(1)
calidad

In [ ]:
resumen_anual = df.groupby("ANIO").agg(accidentes=("ANIO", "size"), muertos=("MUERTOS", "sum"),
                                       heridos=("HERIDOS", "sum"), pct_con_victimas=("CON_VICTIMAS", "mean"))
resumen_anual["pct_con_victimas"] = (resumen_anual["pct_con_victimas"] * 100).round(1)
if 2021 in resumen_anual.index:
    print(f"Chequeo contra INEGI — accidentes 2021: {resumen_anual.loc[2021, 'accidentes']:,} (comunicado oficial: 340,415)")
resumen_anual

## 6. Análisis exploratorio

### 6.1 ¿Cómo ha evolucionado la siniestralidad?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, col, titulo, color in zip(ax, ["accidentes", "heridos", "muertos"],
                                 ["Accidentes", "Personas heridas", "Personas fallecidas"], [AZUL, GRIS, ROJO]):
    a.plot(resumen_anual.index, resumen_anual[col], marker="o", color=color, lw=2)
    a.set_title(titulo); a.set_ylim(0); a.yaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
    a.set_xticks(resumen_anual.index); a.tick_params(axis="x", rotation=45)
fig.suptitle("Siniestralidad vial en zonas urbanas y suburbanas, por año", x=0.01, ha="left", fontweight="bold")
guardar("01_tendencia_anual")

### 6.2 ¿Cuándo ocurren los accidentes y cuándo son más graves?
Dos mapas de calor con la misma cuadrícula (día × hora): el primero muestra **volumen**, el segundo la **proporción de accidentes con víctimas**. Compararlos separa "hay más tráfico" de "es más peligroso".

In [ ]:
t = df.dropna(subset=["HORA", "DIASEMANA"])
vol = t.pivot_table(index="DIASEMANA", columns="HORA", values="ANIO", aggfunc="size", observed=False)
grav = t.pivot_table(index="DIASEMANA", columns="HORA", values="CON_VICTIMAS", aggfunc="mean", observed=False) * 100

fig, ax = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
for a, datos, cmap, titulo, fmt in [(ax[0], vol, "Blues", "Número de accidentes", "{x:,.0f}"),
                                    (ax[1], grav, "Reds", "% de accidentes con víctimas", "{x:.0f}%")]:
    im = a.imshow(datos.values, aspect="auto", cmap=cmap)
    a.set_yticks(range(7)); a.set_yticklabels(datos.index); a.set_title(titulo); a.grid(False)
    fig.colorbar(im, ax=a, format=mtick.StrMethodFormatter(fmt), pad=0.01)
ax[1].set_xticks(range(len(vol.columns))); ax[1].set_xticklabels([int(h) for h in vol.columns]); ax[1].set_xlabel("Hora del día")
guardar("02_dia_hora")

pico_vol = vol.stack().idxmax(); pico_grav = grav.stack().idxmax()
print(f"Mayor volumen: {pico_vol[0]} a las {int(pico_vol[1])} h · Mayor proporción con víctimas: {pico_grav[0]} a las {int(pico_grav[1])} h")

### 6.3 ¿Qué tipos de accidente dejan más víctimas?

In [ ]:
tipo = df.groupby("TIPACCID").agg(accidentes=("ANIO", "size"), pct_con_victimas=("CON_VICTIMAS", "mean"),
                                  muertos_por_mil=("MUERTOS", lambda s: s.sum() / len(s) * 1000))
tipo["pct_con_victimas"] *= 100
tipo = tipo.sort_values("accidentes")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
ax[0].barh(tipo.index, tipo["accidentes"], color=AZUL); ax[0].set_title("Accidentes por tipo")
ax[0].xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax[1].barh(tipo.index, tipo["pct_con_victimas"], color=ROJO); ax[1].set_title("% con víctimas")
ax[1].xaxis.set_major_formatter(mtick.PercentFormatter())
for a in ax: a.grid(axis="y", visible=False)
guardar("03_tipo_accidente")
tipo.sort_values("accidentes", ascending=False).round(1)

### 6.4 ¿Dónde? Comparación entre entidades
Sin datos de población no se pueden calcular tasas por habitante (fuera de alcance), así que se comparan dos cosas que sí son comparables entre estados: el volumen y la **proporción de accidentes con víctimas**. Ojo: una proporción alta también puede reflejar que en esa entidad se registran menos los choques leves.

In [ ]:
ent = df.groupby("ENTIDAD").agg(accidentes=("ANIO", "size"), pct_con_victimas=("CON_VICTIMAS", "mean"),
                                muertos_por_mil=("MUERTOS", lambda s: s.sum() / len(s) * 1000))
ent["pct_con_victimas"] *= 100
ent = ent.sort_values("pct_con_victimas")

fig, ax = plt.subplots(1, 2, figsize=(12, 8), sharey=True)
ax[0].barh(ent.index, ent["accidentes"], color=AZUL); ax[0].set_title("Accidentes registrados")
ax[0].xaxis.set_major_formatter(mtick.StrMethodFormatter("{x:,.0f}"))
ax[1].barh(ent.index, ent["pct_con_victimas"], color=ROJO); ax[1].set_title("% de accidentes con víctimas")
ax[1].axvline(df["CON_VICTIMAS"].mean() * 100, color="black", ls="--", lw=1)
ax[1].text(df["CON_VICTIMAS"].mean() * 100, -1.2, " nacional", fontsize=8, va="center")
ax[1].xaxis.set_major_formatter(mtick.PercentFormatter())
for a in ax: a.grid(axis="y", visible=False)
guardar("04_entidades")

### 6.5 Alcohol, cinturón y motocicletas

In [ ]:
def tasa(col):
    g = df.groupby(col, observed=True).agg(accidentes=("ANIO", "size"), pct_con_victimas=("CON_VICTIMAS", "mean"),
                                           muertos_por_mil=("MUERTOS", lambda s: s.sum() / len(s) * 1000))
    g["pct_con_victimas"] *= 100
    return g.round(1)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for a, col, titulo in zip(ax, ["ALIENTO", "CINTURON", "MOTO"],
                          ["Aliento alcohólico", "Uso de cinturón", "Motocicleta involucrada"]):
    g = tasa(col)
    if col == "MOTO": g.index = g.index.map({0: "No", 1: "Sí"})
    g = g.reindex([i for i in ["Sí", "No", "Se ignora"] if i in g.index])
    barras = a.bar(g.index, g["pct_con_victimas"], color=[ROJO if i == "Sí" else GRIS for i in g.index])
    a.bar_label(barras, fmt="%.0f%%", fontsize=9); a.set_title(titulo); a.grid(axis="x", visible=False)
ax[0].set_ylabel("% de accidentes con víctimas"); ax[0].yaxis.set_major_formatter(mtick.PercentFormatter())
guardar("05_factores")

moto_anual = df.groupby("ANIO")["MOTO"].mean().mul(100).round(1)
print("Porcentaje de accidentes con motocicleta involucrada, por año:"); print(moto_anual.to_string())

## 7. Modelo: ¿qué factores se asocian con que haya víctimas?

Las gráficas anteriores miran un factor a la vez, pero los factores se mezclan (por ejemplo, de madrugada hay más alcohol). Una **regresión logística** estima la asociación de cada factor *manteniendo constantes los demás*, y sus coeficientes se leen como razones de momios (*odds ratios*).

Decisiones de diseño:
- **Variable objetivo:** accidente con víctimas (1) vs. solo daños (0). Los accidentes fatales son muy pocos para un primer modelo estable.
- **Validación temporal:** se entrena con todos los años menos el último y se evalúa en el último. Es más honesto que una partición aleatoria, porque simula usar el modelo sobre un año nuevo.
- **"Se ignora" se conserva como categoría** en vez de eliminar filas: perder esos registros sesgaría la muestra hacia accidentes mejor documentados.
- **Interpretación:** asociaciones, no causas. La base solo contiene accidentes *registrados* y varias variables se capturan después del hecho.

In [ ]:
VARS = ["TIPACCID", "ZONA", "FRANJA", "FIN_DE_SEMANA", "ALIENTO", "CINTURON", "SEXO", "CAPAROD", "MOTO", "GRUPO_EDAD"]
m = df[VARS + ["CON_VICTIMAS", "ANIO"]].copy()
for c in ["MOTO", "FIN_DE_SEMANA"]:
    m[c] = m[c].map({0: "No", 1: "Sí"})
for c in VARS:
    m[c] = m[c].astype(object).where(m[c].notna(), "No especificado").astype(str)

ultimo = m["ANIO"].max()
train, test = m[m["ANIO"] < ultimo], m[m["ANIO"] == ultimo]
if len(train) > 400_000:                               # muestra para que entrene en segundos
    train = train.sample(400_000, random_state=42)

# categoría de referencia = la más frecuente de cada variable (los momios se leen contra ella)
referencias = [train[c].mode()[0] for c in VARS]
modelo = Pipeline([
    ("prep", ColumnTransformer([("oh", OneHotEncoder(drop=referencias, handle_unknown="ignore"), VARS)])),
    ("logit", LogisticRegression(max_iter=2000, C=1.0)),
])
modelo.fit(train[VARS], train["CON_VICTIMAS"])

p = modelo.predict_proba(test[VARS])[:, 1]
auc = roc_auc_score(test["CON_VICTIMAS"], p); ap = average_precision_score(test["CON_VICTIMAS"], p)
base = test["CON_VICTIMAS"].mean()
print(f"Entrenamiento: {len(train):,} accidentes (< {ultimo}) · Prueba: {len(test):,} accidentes ({ultimo})")
print(f"ROC AUC = {auc:.3f} (azar = 0.500) · Precisión promedio = {ap:.3f} (línea base = {base:.3f})")

In [ ]:
nombres = modelo.named_steps["prep"].get_feature_names_out()
momios = pd.Series(np.exp(modelo.named_steps["logit"].coef_[0]), index=[n.replace("oh__", "") for n in nombres])
momios = momios[~momios.index.str.contains("No especificado")].sort_values()
extremos = pd.concat([momios.head(8), momios.tail(12)])

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(extremos.index, extremos.values, color=[ROJO if v > 1 else AZUL for v in extremos.values])
ax.axvline(1, color="black", lw=1); ax.set_xscale("log"); ax.grid(axis="y", visible=False)
ax.set_xticks([t for t in [0.25, 0.5, 1, 2, 4, 8, 16] if extremos.min() / 1.5 <= t <= extremos.max() * 1.5])
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:g}×")); ax.xaxis.set_minor_formatter(mtick.NullFormatter())
ax.set_title("Razón de momios de tener víctimas (frente a la categoría más frecuente de cada variable)")
guardar("06_momios")
print("Categorías de referencia:", dict(zip(VARS, referencias)))

## 8. Resultados

La siguiente celda arma el resumen de hallazgos **a partir de los datos** (no está escrito a mano), y lo guarda en `RESULTADOS.md` para pegarlo en el README del repositorio. Así, si el INEGI publica un año nuevo, basta volver a ejecutar el notebook.

In [ ]:
a0, a1 = resumen_anual.index.min(), resumen_anual.index.max()
var = lambda col: (resumen_anual.loc[a1, col] / resumen_anual.loc[a0, col] - 1) * 100
tipo_frec = tipo["accidentes"].idxmax()
tipo_grave = tipo[tipo["accidentes"] >= 1000]["pct_con_victimas"].idxmax()
f = {c: tasa(c)["pct_con_victimas"] for c in ["ALIENTO", "CINTURON", "MOTO"]}
top_momios = momios.tail(3)[::-1]

resultados = f'''### Resultados principales ({a0}–{a1})

- Se analizaron **{len(df):,} accidentes** con **{df["HERIDOS"].sum():,} personas heridas** y **{df["MUERTOS"].sum():,} fallecidas**; el {df["CON_VICTIMAS"].mean()*100:.1f}% de los accidentes dejó víctimas.
- Entre {a0} y {a1} los accidentes variaron {var("accidentes"):+.1f}%, las personas heridas {var("heridos"):+.1f}% y las fallecidas {var("muertos"):+.1f}%.
- El mayor volumen ocurre el **{pico_vol[0].lower()} a las {int(pico_vol[1])} h**, pero la mayor proporción de accidentes con víctimas se da el **{pico_grav[0].lower()} a las {int(pico_grav[1])} h** ({grav.stack().max():.0f}%).
- El tipo más frecuente es *{tipo_frec.lower()}* ({tipo.loc[tipo_frec, "accidentes"]/len(df)*100:.0f}% del total); el que más víctimas deja es *{tipo_grave.lower()}* ({tipo.loc[tipo_grave, "pct_con_victimas"]:.0f}% con víctimas).
- La participación de motocicletas pasó de {moto_anual.loc[a0]:.1f}% a {moto_anual.loc[a1]:.1f}% de los accidentes; {f["MOTO"].loc[1]:.0f}% de los accidentes con moto dejan víctimas frente a {f["MOTO"].loc[0]:.0f}% del resto.
- Con aliento alcohólico, {f["ALIENTO"].get("Sí", float("nan")):.0f}% de los accidentes deja víctimas (sin aliento: {f["ALIENTO"].get("No", float("nan")):.0f}%); sin cinturón, {f["CINTURON"].get("No", float("nan")):.0f}% (con cinturón: {f["CINTURON"].get("Sí", float("nan")):.0f}%).
- **{ent.index[-1]}** tiene la mayor proporción de accidentes con víctimas ({ent["pct_con_victimas"].iloc[-1]:.0f}%) y **{ent.index[0]}** la menor ({ent["pct_con_victimas"].iloc[0]:.0f}%).
- El modelo logístico, evaluado en {ultimo} (año no visto), obtiene **ROC AUC = {auc:.2f}**. Controlando por los demás factores, los más asociados con víctimas son: {"; ".join(f"{k} ({v:.1f}×)" for k, v in top_momios.items())}.
'''
Path("RESULTADOS.md").write_text(resultados, encoding="utf-8")
from IPython.display import Markdown; Markdown(resultados)

## 9. Documentación del proceso

### Decisiones tomadas
| Decisión | Alternativa descartada | Motivo |
|---|---|---|
| Analizar desde 2015 | Serie completa desde 1997 | Menos memoria y criterios de captación más homogéneos; una década basta para ver tendencia |
| Calcular la gravedad a partir de los conteos de víctimas | Usar la columna de clasificación del INEGI | La columna no viene igual en todas las ediciones; los conteos sí, y el cálculo es transparente |
| Objetivo del modelo: "con víctimas" | "Accidente fatal" | Los fatales son alrededor de 1% de los casos: clases muy desbalanceadas para un primer modelo |
| Validación temporal (último año como prueba) | Partición aleatoria | Simula el uso real y evita optimismo por mezclar años |
| Mantener "Se ignora" como categoría | Eliminar esas filas | Eliminarlas descarta gran parte de la base y sesga hacia accidentes mejor documentados |
| Regresión logística | Bosques aleatorios / boosting | El fin es explicar, no maximizar la predicción; los momios se comunican fácilmente |
| Comparar entidades por % con víctimas | Tasas por 100 mil habitantes | Requiere otra fuente (población); queda como trabajo futuro |

### Desafíos y cómo se resolvieron
- **Esquemas distintos entre productos y ediciones del INEGI** (nombres de columnas, claves numéricas vs. etiquetas, codificación de caracteres). → Capa de normalización: renombrado de columnas, función `a_etiqueta` que detecta el formato y prueba de dos codificaciones al leer.
- **Filas que no son accidentes** ("certificado cero") **y valores centinela** (99, 0, 32) que distorsionan promedios y conteos. → Se leyó el descriptor de archivos antes de analizar, se eliminaron los certificados y los centinelas pasaron a valor faltante.
- **Variables de conducta con mucho "se ignora"** (alcohol, cinturón). → Se cuantifica en la tabla de calidad, se conserva como categoría y se advierte en la interpretación.
- **Confundir volumen con riesgo.** Las horas con más choques no son las más peligrosas. → Dos mapas de calor con la misma cuadrícula y un modelo que controla por varios factores a la vez.
- **Reproducibilidad.** → Descarga automática desde la fuente oficial, semillas fijas y resultados generados por código.

> ✏️ **Personaliza esta sección** con lo que te ocurra al ejecutarlo: errores que encontraste, cifras que te sorprendieron y qué cambiaste a raíz de ello. Es lo que más valoran quienes revisan portafolios.

### Limitaciones
- ATUS cubre zonas urbanas y suburbanas; no incluye carreteras federales.
- Solo hay accidentes **registrados** por las autoridades locales; el subregistro de choques leves varía entre municipios y afecta las comparaciones entre entidades.
- Las víctimas se cuentan en el lugar del hecho; no incluye fallecimientos posteriores en hospital.
- Los resultados del modelo son asociaciones, no efectos causales.

### Trabajo futuro
- Tasas por 100 mil habitantes y por parque vehicular (CONAPO, vehículos de motor registrados del INEGI).
- Análisis municipal y mapas con la información georreferenciada de ATUS.
- Modelo específico para accidentes fatales con técnicas para clases desbalanceadas.
- Tablero interactivo (Streamlit o Power BI) para explorar por entidad.

## 10. Publicación en el portafolio

Estructura sugerida del repositorio en GitHub:

```
accidentes-transito-inegi/
├── README.md                      ← descripción breve (plantilla abajo) + contenido de RESULTADOS.md
├── accidentes_transito_inegi.ipynb
├── requirements.txt               ← pandas, numpy, matplotlib, scikit-learn, jupyter
├── figuras/                       ← se generan al ejecutar el notebook
└── .gitignore                     ← incluir datos/ (el ZIP se descarga solo; no hace falta subirlo)
```

**Descripción breve para el README / LinkedIn**

> **Siniestralidad vial en México (INEGI-ATUS, 2015 en adelante)**
> **Objetivo:** identificar cuándo, dónde y bajo qué condiciones los accidentes de tránsito dejan víctimas, a partir de los datos abiertos del INEGI.
> **Mi rol:** proyecto individual; definí el problema y el alcance, construí el proceso reproducible de descarga y limpieza en Python, hice el análisis exploratorio, ajusté un modelo de regresión logística con validación temporal y comuniqué los hallazgos.
> **Resultados:** *pega aquí 3 o 4 viñetas de `RESULTADOS.md`.*
> **Tecnologías:** Python, pandas, matplotlib, scikit-learn, Jupyter.

**Fuente de los datos:** INEGI. Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS). <https://www.inegi.org.mx/programas/accidentes/>